# DRL Assignment 3: Meta and Transfer Learning

**Setup for Google Colab with GPU acceleration**

Before running:
1. Go to Runtime → Change runtime type → Select **GPU** (T4, A100, or V100)
2. For best performance, select **A100** if available with your processor units

In [10]:
# Detect platform (Colab vs Kaggle vs other)
import os
try:
    IN_COLAB = 'COLAB_GPU' in os.environ or 'google.colab' in str(get_ipython())
except:
    IN_COLAB = False
IN_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ

print(f"Running on: {'Colab' if IN_COLAB else 'Kaggle' if IN_KAGGLE else 'Other'}")

# Clone/update repository
!git clone https://github.com/omereliy/DRL-ass3.git 2>/dev/null || (cd DRL-ass3 && git pull)
%cd DRL-ass3

# Uninstall old gym to avoid NumPy 2.0 conflicts
%pip uninstall -y gym 2>/dev/null || true

# Install dependencies with pinned gymnasium version (0.30.0+ fixes np.float_ issue)
%pip install -q "gymnasium>=0.30.0" torch tensorboard numpy

Running on: Kaggle
/kaggle/working/DRL-ass3/DRL-ass3
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Check GPU availability
!nvidia-smi

In [ ]:
# Verify GPU is available for PyTorch
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
# Import modules with platform-agnostic path
import sys
import os

# Set base path based on platform
if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
    base_path = '/kaggle/working/DRL-ass3'
else:
    base_path = '/content/DRL-ass3'

sys.path.insert(0, base_path)

from src.utils import TrainingConfig, DEVICE
from src.actor_critic import train_individual_network
from src.fine_tuning import run_section2_experiments
from src.progressive_networks import run_section3_experiments

print(f"Using device: {DEVICE}")

## Optimized Configuration for Speed

Key optimizations:
- Higher learning rate for faster convergence
- Larger hidden dimensions leverage GPU parallelism
- Reduced log interval to minimize I/O overhead

In [ ]:
# Optimized configuration for faster training
fast_config = TrainingConfig(
    gamma=0.99,
    lr_actor=3e-3,      # Higher LR for faster convergence
    lr_critic=3e-3,
    hidden_dim=256,     # Larger network - GPUs handle this well
    max_episodes=1500,  # Usually converges before this
    max_steps=500,
    entropy_coef=0.01,
    value_loss_coef=0.5,
    log_interval=50,    # Less frequent logging
    save_interval=200,
    seed=42
)

## Section 1: Train Individual Networks

Train actor-critic networks for:
- CartPole-v1
- Acrobot-v1  
- MountainCarContinuous-v0

In [ ]:
%%time
# Train CartPole
print("="*60)
print("Training CartPole-v1")
print("="*60)
cartpole_stats = train_individual_network("CartPole-v1", fast_config)

In [ ]:
%%time
# Train Acrobot
print("="*60)
print("Training Acrobot-v1")
print("="*60)
acrobot_stats = train_individual_network("Acrobot-v1", fast_config)

In [ ]:
%%time
# Train MountainCar
print("="*60)
print("Training MountainCarContinuous-v0")
print("="*60)
mountaincar_stats = train_individual_network("MountainCarContinuous-v0", fast_config)

In [ ]:
# Section 1 Summary
print("\n" + "="*70)
print("SECTION 1 RESULTS")
print("="*70)
section1_results = {
    "CartPole-v1": cartpole_stats,
    "Acrobot-v1": acrobot_stats,
    "MountainCarContinuous-v0": mountaincar_stats
}

print(f"{'Environment':<30} {'Episodes':<12} {'Time (s)':<12} {'Converged':<12} {'Avg Reward':<15}")
print("-"*80)
for env, stats in section1_results.items():
    conv = stats.convergence_episode if stats.convergence_episode else "N/A"
    print(f"{env:<30} {stats.total_episodes:<12} {stats.training_time:<12.2f} {str(conv):<12} {stats.final_avg_reward:<15.2f}")

## Section 2: Fine-tuning

Transfer learning via fine-tuning:
1. Acrobot → CartPole
2. CartPole → MountainCar

In [ ]:
%%time
# Run Section 2 experiments
section2_results = run_section2_experiments(fast_config)

In [ ]:
# Section 2 Summary
print("\n" + "="*70)
print("SECTION 2 RESULTS - Fine-tuning")
print("="*70)
print(f"{'Transfer':<35} {'Episodes':<12} {'Time (s)':<12} {'Converged':<12} {'Avg Reward':<15}")
print("-"*85)
for transfer, stats in section2_results.items():
    conv = stats.convergence_episode if stats.convergence_episode else "N/A"
    print(f"{transfer:<35} {stats.total_episodes:<12} {stats.training_time:<12.2f} {str(conv):<12} {stats.final_avg_reward:<15.2f}")

## Section 3: Progressive Networks

Transfer from multiple sources:
1. {Acrobot, MountainCar} → CartPole
2. {CartPole, Acrobot} → MountainCar

In [ ]:
%%time
# Run Section 3 experiments
section3_results = run_section3_experiments(fast_config)

In [ ]:
# Section 3 Summary
print("\n" + "="*70)
print("SECTION 3 RESULTS - Progressive Networks")
print("="*70)
print(f"{'Transfer':<40} {'Episodes':<12} {'Time (s)':<12} {'Converged':<12} {'Avg Reward':<15}")
print("-"*90)
for transfer, stats in section3_results.items():
    conv = stats.convergence_episode if stats.convergence_episode else "N/A"
    print(f"{transfer:<40} {stats.total_episodes:<12} {stats.training_time:<12.2f} {str(conv):<12} {stats.final_avg_reward:<15.2f}")

## Final Comparison

In [ ]:
# Compare results across all sections
print("\n" + "="*90)
print("COMPLETE RESULTS COMPARISON")
print("="*90)

# Compare CartPole training
print("\n--- CartPole Training Comparison ---")
print(f"{'Method':<45} {'Episodes':<12} {'Time (s)':<12}")
print("-"*70)
print(f"{'Section 1: From scratch':<45} {cartpole_stats.total_episodes:<12} {cartpole_stats.training_time:<12.2f}")
print(f"{'Section 2: Fine-tuned from Acrobot':<45} {section2_results['acrobot_to_cartpole'].total_episodes:<12} {section2_results['acrobot_to_cartpole'].training_time:<12.2f}")
print(f"{'Section 3: Progressive (Acrobot+MountainCar)':<45} {section3_results['acrobot_mountaincar_to_cartpole'].total_episodes:<12} {section3_results['acrobot_mountaincar_to_cartpole'].training_time:<12.2f}")

# Compare MountainCar training
print("\n--- MountainCar Training Comparison ---")
print(f"{'Method':<45} {'Episodes':<12} {'Time (s)':<12}")
print("-"*70)
print(f"{'Section 1: From scratch':<45} {mountaincar_stats.total_episodes:<12} {mountaincar_stats.training_time:<12.2f}")
print(f"{'Section 2: Fine-tuned from CartPole':<45} {section2_results['cartpole_to_mountaincar'].total_episodes:<12} {section2_results['cartpole_to_mountaincar'].training_time:<12.2f}")
print(f"{'Section 3: Progressive (CartPole+Acrobot)':<45} {section3_results['cartpole_acrobot_to_mountaincar'].total_episodes:<12} {section3_results['cartpole_acrobot_to_mountaincar'].training_time:<12.2f}")

## TensorBoard Visualization

In [ ]:
# Load TensorBoard
%load_ext tensorboard
%tensorboard --logdir logs/

## Plot Learning Curves

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def moving_avg(data, window=100):
    if len(data) < window:
        return data
    cumsum = np.cumsum(np.insert(data, 0, 0))
    return (cumsum[window:] - cumsum[:-window]) / window

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Section 1 plots
for idx, (env, stats) in enumerate(section1_results.items()):
    ax = axes[0, idx]
    rewards = stats.rewards_history
    ax.plot(rewards, alpha=0.3, label='Episode Reward')
    if len(rewards) >= 100:
        ax.plot(range(99, len(rewards)), moving_avg(rewards), label='Moving Avg (100)')
    ax.set_title(f'Section 1: {env.split("-")[0]}')
    ax.set_xlabel('Episode')
    ax.set_ylabel('Reward')
    ax.legend()
    ax.grid(True, alpha=0.3)

# Section 2 & 3 comparison for CartPole
ax = axes[1, 0]
for label, stats in [('From Scratch', cartpole_stats),
                      ('Fine-tuned', section2_results['acrobot_to_cartpole']),
                      ('Progressive', section3_results['acrobot_mountaincar_to_cartpole'])]:
    rewards = stats.rewards_history
    if len(rewards) >= 100:
        ax.plot(range(99, len(rewards)), moving_avg(rewards), label=label)
ax.set_title('CartPole: Transfer Learning Comparison')
ax.set_xlabel('Episode')
ax.set_ylabel('Reward (Moving Avg)')
ax.legend()
ax.grid(True, alpha=0.3)

# Section 2 & 3 comparison for MountainCar
ax = axes[1, 1]
for label, stats in [('From Scratch', mountaincar_stats),
                      ('Fine-tuned', section2_results['cartpole_to_mountaincar']),
                      ('Progressive', section3_results['cartpole_acrobot_to_mountaincar'])]:
    rewards = stats.rewards_history
    if len(rewards) >= 100:
        ax.plot(range(99, len(rewards)), moving_avg(rewards), label=label)
ax.set_title('MountainCar: Transfer Learning Comparison')
ax.set_xlabel('Episode')
ax.set_ylabel('Reward (Moving Avg)')
ax.legend()
ax.grid(True, alpha=0.3)

# Training time comparison
ax = axes[1, 2]
methods = ['Scratch', 'Fine-tune', 'Progressive']
cartpole_times = [cartpole_stats.training_time,
                  section2_results['acrobot_to_cartpole'].training_time,
                  section3_results['acrobot_mountaincar_to_cartpole'].training_time]
mountaincar_times = [mountaincar_stats.training_time,
                     section2_results['cartpole_to_mountaincar'].training_time,
                     section3_results['cartpole_acrobot_to_mountaincar'].training_time]

x = np.arange(len(methods))
width = 0.35
ax.bar(x - width/2, cartpole_times, width, label='CartPole')
ax.bar(x + width/2, mountaincar_times, width, label='MountainCar')
ax.set_ylabel('Training Time (s)')
ax.set_title('Training Time Comparison')
ax.set_xticks(x)
ax.set_xticklabels(methods)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('report/learning_curves.png', dpi=150)
plt.show()

## Save Results

In [ ]:
import json
from datetime import datetime

def stats_to_dict(stats):
    return {
        'total_episodes': stats.total_episodes,
        'total_steps': stats.total_steps,
        'training_time': stats.training_time,
        'final_avg_reward': stats.final_avg_reward,
        'convergence_episode': stats.convergence_episode
    }

results = {
    'timestamp': datetime.now().isoformat(),
    'section1': {env: stats_to_dict(s) for env, s in section1_results.items()},
    'section2': {k: stats_to_dict(s) for k, s in section2_results.items()},
    'section3': {k: stats_to_dict(s) for k, s in section3_results.items()}
}

with open('report/results.json', 'w') as f:
    json.dump(results, f, indent=2)

print("Results saved to report/results.json")

In [ ]:
# Download results
from google.colab import files

# Zip everything
!zip -r drl_assignment3_results.zip models/ logs/ report/
files.download('drl_assignment3_results.zip')